In [10]:
from dotenv import load_dotenv
load_dotenv()

True

# 预定义中间件

## PIIMiddleware-个人信息脱敏
可以在调用模型前后自动检测并脱敏输入、输出消息中的个人身份信息（PII），如邮箱、电话号码、身份证号等。

PII脱敏处理策略有四种：  
- 'block' - 抛出异常
- 'redact' - 用 [REDACTED_{PII_TYPE}] 来替代
- 'mask' - 关键信息采用**掩码 (例如., ****-****-****-1234)
- 'hash' - 用哈希值来替换

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

# 可添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  # 掩码
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="block",
    apply_to_input=True
)

# 在agent中注册
agent_with_builtin_middleware = create_agent(
    model = "deepseek-chat",
    middleware=[pii_middleware_email, pii_middleware_phone]
)

# 测试
response = agent_with_builtin_middleware.invoke(
    {"messages":[HumanMessage("我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")]}
)

for m in response["messages"]:
    m.pretty_print()

PIIDetectionError: Detected 1 instance(s) of phone_number in text content

## ModelFallbackMiddleware
模型调用失败时给出降级处理方案。可以在创建时设置多个模型，如果主模型调用失败，会自动调用备用模型。

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

# 设置主模型和备用模型
model_fallback_middleware = ModelFallbackMiddleware(
    "gpt-4o-mini",  # 默认模型
    "deepseek-chat"
)

agent_with_model_fallback = create_agent(
    model="gpt-4o-mini",
    middleware=[model_fallback_middleware]
)

# 测试
response = agent_with_model_fallback.invoke({
    "messages": [HumanMessage("你好")]
})
for m in response['messages']:
    m.pretty_print()

## HumanInTheLoopMiddleware-人工审核
让人工介入到Agent执行流程中，在执行工具调用前暂停，等待人工确认。  
通常用于Agent执行敏感操作前的确认

In [14]:
from langchain.tools import tool

@tool
def transfer_money(amount: int, to: str):
    """
    向指定账户转账
    arg:
        amount: 金额
        to: 收款人
    """
    return f"已向{to}转账{amount}元"

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "transfer_money": {
            "description": "请确认转账操作",
            "allowed_decisions": ["approve", 'reject', 'edit']
        }
    }
)